<a href="https://colab.research.google.com/github/benhamad-mhemed/pfe-master/blob/main/Deep_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Partie 3 : Traitement et Apprentissage Deep Learning

## 1- Importation des Bibliothèques

In [1]:
# Installation et importation des bibliothèques
# =====================================
!pip install tensorflow --quiet

import os
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Connexion Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2- Chargement du dataset

In [2]:
df = pd.read_excel("/content/drive/MyDrive/Master/Scolarite/resultats/df_1A.xlsx")

## 3- Préparation des features et target

In [3]:
X = df.drop(columns=["Orientation"])  # features
y = df["Orientation"]                 # target

### Encodage

In [4]:
le = LabelEncoder()
y = le.fit_transform(y)

### Séparation train/test

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

### Normalisation

In [6]:
scaler = StandardScaler()
X_train_dl = scaler.fit_transform(X_train)
X_test_dl = scaler.transform(X_test)

## 4-  Création d'un nouvel élève pour prédiction

In [7]:
nouvel_eleve = pd.DataFrame([[
    0,          # Genre
    20,         # Age
    0,          # Type_Hand
    1,          # Etat_Fam
    1,          # Type_Res
    2,          # Jours_Abs
    3,          # Jours_Sanc
    0,          # Nbr_Ann_Red
    14,         # Arabe
    12,         # Français
    13,         # Anglais
    15,         # Math
    17,         # Sc_Phys
    10,         # SVT
    18,         # Techno
    18,         # Info
    11,         # Hist_Geo
    14,         # Moy_Ann
    1           # Orientation (valeur fictive, peut être ignorée pour prédiction)
]], columns=X.columns)

nouvel_eleve_scaled = scaler.transform(nouvel_eleve)


## 5- Définition des sections pour interprétation

In [8]:
sections = {
    1: "section lettre",
    2: "section science",
    3: "section technologie informatique",
    4: "section economie et service",
    5: "non orienté"
}

## 6- Modèle 1 : ANN simple

### Création du modèle

In [9]:
model_ann = keras.Sequential([

    layers.Dense(64, activation="relu", input_shape=(X_train.shape[1],)),

    layers.Dense(32, activation="relu"),

    layers.Dense(16, activation="relu"),

    layers.Dense(len(np.unique(y)), activation="softmax")
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


### Compilation

In [10]:
model_ann.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

### Entraînement

In [11]:
history_ann = model_ann.fit(
    X_train_dl,
    y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2
)

Epoch 1/50
41/41 ━━━━━━━━━━━━━━━━━━━━ 5s 32ms/step - accuracy: 0.3344 - loss: 1.4540 - val_accuracy: 0.5046 - val_loss: 1.2535
Epoch 2/50
41/41 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.5100 - loss: 1.1743 - val_accuracy: 0.6062 - val_loss: 1.0923
Epoch 3/50
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.5716 - loss: 1.0464 - val_accuracy: 0.6277 - val_loss: 1.0320
Epoch 4/50
41/41 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.5986 - loss: 0.9726 - val_accuracy: 0.6092 - val_loss: 0.9854
Epoch 5/50
41/41 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.6294 - loss: 0.9189 - val_accuracy: 0.6092 - val_loss: 0.9680
Epoch 6/50
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.6379 - loss: 0.8827 - val_accuracy: 0.6123 - val_loss: 0.9612
Epoch 7/50
41/41 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.6410 - loss: 0.8568 - val_accuracy: 0.6031 - val_loss: 0.9510
Epoch 8/50
41/41 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.6525 - loss: 0.8323 - val_accuracy: 0.6277 - val_

### Accuracy

In [12]:
loss_ann, acc_ann = model_ann.evaluate(X_test_dl, y_test)

print("ANN simple Accuracy :", acc_ann)

13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5961 - loss: 1.1070 
ANN simple Accuracy : 0.5960590839385986


## 7- Modèle 2 : ANN avec Dropout

### Création du modèle

In [13]:
model_dropout = keras.Sequential([

    layers.Dense(128, activation="relu", input_shape=(X_train.shape[1],)),
    layers.Dropout(0.3),

    layers.Dense(64, activation="relu"),
    layers.Dropout(0.3),

    layers.Dense(32, activation="relu"),

    layers.Dense(len(np.unique(y)), activation="softmax")
])

### Compilation

In [14]:
model_dropout.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

### Entraînement

In [15]:
history_dropout = model_dropout.fit(
    X_train_dl,
    y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2
)

Epoch 1/50
41/41 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.4106 - loss: 1.3981 - val_accuracy: 0.5754 - val_loss: 1.1605
Epoch 2/50
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5239 - loss: 1.1519 - val_accuracy: 0.5815 - val_loss: 1.0323
Epoch 3/50
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5593 - loss: 1.0502 - val_accuracy: 0.6123 - val_loss: 0.9736
Epoch 4/50
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5709 - loss: 1.0076 - val_accuracy: 0.6031 - val_loss: 0.9544
Epoch 5/50
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5840 - loss: 0.9573 - val_accuracy: 0.5908 - val_loss: 0.9600
Epoch 6/50
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6179 - loss: 0.9254 - val_accuracy: 0.6031 - val_loss: 0.9357
Epoch 7/50
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6140 - loss: 0.9126 - val_accuracy: 0.6031 - val_loss: 0.9403
Epoch 8/50
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6140 - loss: 0.9025 - val_accuracy: 0.6123 - val_loss:

### Accuracy

In [16]:
loss_dropout, acc_dropout = model_dropout.evaluate(X_test_dl, y_test)

print("ANN Dropout Accuracy :", acc_dropout)

13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6379 - loss: 0.9136 
ANN Dropout Accuracy : 0.6379310488700867


## 8- Prédiction pour le nouvel élève

### Prédiction avec ANN simple

In [17]:
prediction_ann = model_ann.predict(nouvel_eleve_scaled)
classe_ann = np.argmax(prediction_ann)
pred_val_ann = le.inverse_transform([classe_ann])[0]
print("ANN simple - Orientation prédite :", pred_val_ann)
section_orientee = sections.get(pred_val_ann, "section inconnue")
print("Section :", section_orientee)
print("Probabilités par section :")
for i, prob in enumerate(prediction_ann[0]):
    classe = le.inverse_transform([i])[0]
    section = sections.get(classe, "section inconnue")
    print(section, ":", round(prob * 100, 2), "%")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step
ANN simple - Orientation prédite : 3
Section : section technologie informatique
Probabilités par section :
section lettre : 0.0 %
section science : 0.1 %
section technologie informatique : 99.9 %
section economie et service : 0.0 %
non orienté : 0.0 %


### Prédiction avec ANN Dropout

In [18]:
prediction_dropout = model_dropout.predict(nouvel_eleve_scaled)
classe_dropout = np.argmax(prediction_dropout)
pred_val_dropout = le.inverse_transform([classe_dropout])[0]
print("ANN Dropout - Orientation prédite :", pred_val_dropout)
section_orientee = sections.get(pred_val_dropout, "section inconnue")
print("Section :", section_orientee)
print("Probabilités par section :")
for i, prob in enumerate(prediction_dropout[0]):
    classe = le.inverse_transform([i])[0]
    section = sections.get(classe, "section inconnue")
    print(section, ":", round(prob * 100, 2), "%")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 122ms/step
ANN Dropout - Orientation prédite : 3
Section : section technologie informatique
Probabilités par section :
section lettre : 0.0 %
section science : 25.28 %
section technologie informatique : 74.34 %
section economie et service : 0.38 %
non orienté : 0.0 %


### comparaison de deux modèles :

In [19]:
print("ANN :", acc_ann)
print("ANN Dropout :", acc_dropout)

ANN : 0.5960590839385986
ANN Dropout : 0.6379310488700867
